# 09. 데모 서비스 — 모델 비교 챗봇

**작성일**: 2026-09-21
**사용 체크포인트**: `checkpoints/final/` (`final.ipynb` 8:1:1 실행 결과)

질문 하나를 **세 모델에 동시에** 넣어 답변을 나란히 보여주는 웹 데모다. 그냥 챗봇을 만들지 않고 비교 형태로 만든 이유는, 이 프로젝트의 결론 세 가지를 **말이 아니라 화면으로** 보여줄 수 있기 때문이다.

| 이 데모가 보여주는 것 | 어디서 나온 결론인가 |
|---|---|
| 전이학습이 베이스라인보다 낫다 | 5단계 — BLEU 4.87 → 7.28 |
| EOS 결함 모델은 문장을 끝내지 못한다 | 4단계 — 정상 종결률 19.0% → 97.5% |
| 베이스라인에서 greedy는 반복 붕괴를 일으킨다 | 6단계 — 연속반복률 0.121 → 0.014 (beam5) |

**비교 대상 3종**

| 카드 | 체크포인트 | 성격 | BLEU |
|---|---|---|---|
| 베이스라인 | `baseline_best.pt` | 처음부터 학습한 Transformer (6.0M) | 4.87 |
| KoBART — EOS 결함 | `kobart_naive/` | 학습 정답에 EOS를 안 붙인 버전 (124M) | 5.21 |
| KoBART — 최종 | `kobart_B1_lr2e-5/` | EOS 수정 + lr 2e-5 (124M) | 7.28 |

> **핵심**: 가운데와 오른쪽은 **학습 코드에서 정답 끝에 EOS를 붙이는 한 줄**만 다르다. 나머지 조건(모델·lr·배치·epoch·시드·데이터)은 완전히 동일하다.

**실행 방법**: 이 노트북을 위에서부터 실행하면 마지막 셀에서 `http://127.0.0.1:7860` 으로 서버가 뜬다. 노트북 커널이 살아 있는 동안 동작한다.

In [1]:
# 2026-09-21: 공통 준비 - final.ipynb와 같은 경로/디바이스 설정
import math
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

PROC = Path('../dataset/processed')
CKPT = Path('../checkpoints/final')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 베이스라인은 02단계에서 직접 학습한 BPE 토크나이저를 쓴다 (전이학습은 각 모델 고유 토크나이저)
sp = spm.SentencePieceProcessor(model_file=str(PROC / 'chatbot_bpe.model'))
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
VOCAB_SIZE, MAX_LEN = sp.get_piece_size(), 16

print('device:', device, '| BPE vocab:', VOCAB_SIZE, '| MAX_LEN:', MAX_LEN)

C:\Users\iymg1\Documents\nlp-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda | BPE vocab: 8000 | MAX_LEN: 16


## 1. 베이스라인 모델 구조 재정의

체크포인트는 가중치(state_dict)만 담고 있으므로, **final.ipynb 3단계와 완전히 같은 구조**를 다시 정의해야 로드된다. 구조가 한 군데라도 다르면 `load_state_dict`가 실패한다.

In [2]:
# 2026-09-21: final.ipynb 3단계와 동일한 구조 (weight tying 포함)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.d_model = d_model

    def forward(self, tokens):
        return self.embedding(tokens) * math.sqrt(self.d_model)


class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, n_enc, n_dec, ff, dropout):
        super().__init__()
        self.tok_emb = TokenEmbedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=n_enc,
            num_decoder_layers=n_dec, dim_feedforward=ff, dropout=dropout,
            batch_first=True)
        self.generator = nn.Linear(d_model, vocab_size)
        self.generator.weight = self.tok_emb.embedding.weight  # weight tying

    def encode(self, src, src_mask):
        return self.transformer.encoder(self.pos_enc(self.tok_emb(src)), src_mask)

    def decode(self, tgt, memory, tgt_mask):
        return self.transformer.decoder(self.pos_enc(self.tok_emb(tgt)), memory, tgt_mask)


def causal_mask(sz):
    m = (torch.triu(torch.ones((sz, sz), device=device)) == 1).transpose(0, 1)
    return m.float().masked_fill(m == 0, float('-inf')).masked_fill(m == 1, 0.0)


def encode_text(t, max_len=MAX_LEN):
    # 2단계 인코딩 전략과 동일: [BOS] + BPE ids + [EOS]
    return [BOS_ID] + sp.encode(t, out_type=int)[: max_len - 2] + [EOS_ID]

print('구조 정의 완료')

구조 정의 완료


## 2. 디코딩 함수

6단계에서 고른 설정을 그대로 쓴다 — 베이스라인은 `beam5(length_penalty=0.7)`, 전이학습은 `beam5 + no_repeat_ngram_size=3`. 데모에서는 비교를 위해 `greedy`도 고를 수 있게 한다.

In [3]:
# 2026-09-21: 베이스라인용 greedy / beam5 (final.ipynb 3·6단계 구현과 동일)
@torch.no_grad()
def baseline_generate(model, q, strategy='beam5'):
    model.eval()
    src = torch.tensor(encode_text(q), dtype=torch.long, device=device).unsqueeze(0)
    memory = model.encode(src, torch.zeros((src.size(1), src.size(1)), device=device))

    if strategy == 'greedy':
        ys = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)
        for _ in range(MAX_LEN - 1):
            out = model.decode(ys, memory, causal_mask(ys.size(1)))
            nxt = model.generator(out[:, -1]).argmax(-1).item()
            ys = torch.cat([ys, torch.tensor([[nxt]], device=device)], 1)
            if nxt == EOS_ID:
                break
        ids = ys.squeeze(0).tolist()[1:]
    else:
        beams = [([BOS_ID], 0.0)]
        for _ in range(MAX_LEN - 1):
            cands = []
            for toks, sc in beams:
                if toks[-1] == EOS_ID:
                    cands.append((toks, sc))
                    continue
                ys = torch.tensor([toks], dtype=torch.long, device=device)
                out = model.decode(ys, memory, causal_mask(ys.size(1)))
                lp = F.log_softmax(model.generator(out[:, -1]), -1).squeeze(0)
                v, i = lp.topk(5)
                cands += [(toks + [int(ix)], sc + float(vv)) for vv, ix in zip(v, i)]
            # length_penalty=0.7 - 짧은 문장 쏠림 보정
            cands.sort(key=lambda x: x[1] / (len(x[0]) ** 0.7), reverse=True)
            beams = cands[:5]
            if all(t[-1] == EOS_ID for t, _ in beams):
                break
        ids = beams[0][0][1:]

    if ids and ids[-1] == EOS_ID:
        ids = ids[:-1]
    return sp.decode(ids)


@torch.no_grad()
def hf_generate(model, tok, q, strategy='beam5'):
    kw = (dict(num_beams=1, do_sample=False) if strategy == 'greedy'
          else dict(num_beams=5, no_repeat_ngram_size=3))
    enc = tok(q, return_tensors='pt').to(device)
    out = model.generate(**enc, max_length=MAX_LEN, **kw)
    return tok.decode(out[0], skip_special_tokens=True)

print('디코딩 함수 준비 완료')

디코딩 함수 준비 완료


## 3. 모델 3종 로드

In [4]:
# 2026-09-21: 비교할 세 모델을 메모리에 올린다 (KoBART 2개 약 1GB, 12GB GPU에 여유)
import time
t0 = time.time()

baseline = Seq2SeqTransformer(VOCAB_SIZE, 256, 8, 3, 3, 512, 0.1).to(device)
baseline.load_state_dict(torch.load(CKPT / 'baseline_best.pt', map_location=device))
baseline.eval()

MODELS = {'baseline': baseline}
for key, folder in (('naive', 'kobart_naive'), ('final', 'kobart_B1_lr2e-5')):
    MODELS[key] = (AutoModelForSeq2SeqLM.from_pretrained(CKPT / folder).to(device).eval(),
                   AutoTokenizer.from_pretrained(CKPT / folder))

# test set 정답을 함께 보여주기 위해 로드 (데모 입력이 test에 있는 질문이면 정답도 표시)
TEST = pd.read_csv(PROC / 'test.csv')
ANSWER_MAP = dict(zip(TEST['Q'], TEST['A']))

print(f'모델 3종 로드 완료 ({time.time() - t0:.1f}초) | test 정답 {len(ANSWER_MAP)}건')

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Loading weights:  85%|████████▌ | 221/260 [00:00<00:00, 2192.85it/s]

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 1820.18it/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Loading weights:  71%|███████   | 185/260 [00:00<00:00, 1843.94it/s]

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 1729.49it/s]

모델 3종 로드 완료 (1.0초) | test 정답 1173건


In [5]:
# 2026-09-21: 세 모델에 같은 질문을 넣어 결과를 모으는 함수
# 정상 종결 판정 규칙은 4단계 진단에서 쓴 것과 동일하다
SENT_END = ('.', '!', '?', '요', '다', '네', '죠', '까', '용')


def answer(q, strategy='beam5'):
    q = (q or '').strip()
    if not q:
        return None
    outs = {
        'baseline': baseline_generate(MODELS['baseline'], q, strategy),
        'naive': hf_generate(*MODELS['naive'], q, strategy),
        'final': hf_generate(*MODELS['final'], q, strategy),
    }
    return {k: {'text': v, 'len': len(v), 'ended': v.strip().endswith(SENT_END)}
            for k, v in outs.items()}

print('준비 완료')

준비 완료


## 4. 동작 확인

UI를 붙이기 전에 세 모델이 제대로 답하는지 확인한다. **가운데(결함) 모델의 `종결` 표시가 X로 나오는지**가 핵심이다.

In [6]:
# 2026-09-21: 샘플 질문으로 동작 확인
for q in ['청첩장을 찍는 날이 오다니', '배고파', '여자친구랑 헤어졌어']:
    r = answer(q)
    print(f'Q: {q}')
    if q in ANSWER_MAP:
        print(f'   [정답]        {ANSWER_MAP[q]}')
    for k, name in (('baseline', '베이스라인'), ('naive', 'KoBART 결함'), ('final', 'KoBART 최종')):
        d = r[k]
        print(f"   [{name:<11}] {d['text']}   ({d['len']}자, 종결 {'O' if d['ended'] else 'X'})")
    print()

Q: 청첩장을 찍는 날이 오다니
   [정답]        결혼 얼마 안 남았나봐요.
   [베이스라인      ] 건 대화를 나눠보세요.   (12자, 종결 O)
   [KoBART 결함  ] 찍는 찍는 재미가 있죠. 찍고 오세요. 찍고   (24자, 종결 X)
   [KoBART 최종  ] 청첩장을 찍어보세요.   (11자, 종결 O)

Q: 배고파
   [정답]        얼른 맛난 음식 드세요.
   [베이스라인      ] 좀 나아졌길 바랍니다.   (12자, 종결 O)
   [KoBART 결함  ] 얼른 뭐 좀 드세요! 기운 내세요! 뭐 좀 챙겨   (26자, 종결 X)
   [KoBART 최종  ] 뭐라도 드세요.   (8자, 종결 O)



Q: 여자친구랑 헤어졌어
   [베이스라인      ] 사랑이었으니까요.   (9자, 종결 O)
   [KoBART 결함  ] 헤어졌는데 연락하는게 쉽지 않죠. 힘들겠어요. 걱정되   (29자, 종결 X)
   [KoBART 최종  ] 좋은 선택이었길 바라요.   (13자, 종결 O)



In [7]:
# 2026-09-21: 디코딩 전략 효과 확인 - 베이스라인에서 greedy는 반복 붕괴를 일으킨다 (6단계 결론)
q = '청첩장을 찍는 날이 오다니'
for strat in ('greedy', 'beam5'):
    d = answer(q, strat)['baseline']
    print(f"[베이스라인 / {strat:<6}] {d['text']}   ({d['len']}자, 종결 {'O' if d['ended'] else 'X'})")

[베이스라인 / greedy] 그렇게 들어서 들어서 들어서 들어서 들어서 들어서 들어서 들어서 들어서 들어서 들어서 들어서 들어서 들어서   (59자, 종결 X)


[베이스라인 / beam5 ] 건 대화를 나눠보세요.   (12자, 종결 O)


## 5. Gradio UI

**`queue=False`를 쓰는 이유**: Gradio는 기본적으로 SSE(`queue/data`) 스트림으로 결과를 돌려주는데, 일부 내장 브라우저에서 이 연결이 끊긴다. 생성이 1~2초로 짧고 로컬 단일 사용자 데모라 큐가 필요 없으므로, 동기 POST 엔드포인트를 쓰도록 했다.

**색을 직접 지정한 이유**: 답변 상자에 배경색을 칠하므로 글자색도 함께 지정해야 한다. 안 그러면 다크 모드에서 흰 글자가 밝은 배경에 찍혀 보이지 않는다.

In [8]:
# 2026-09-21: 카드 렌더링 - 모델별 답변을 색으로 구분한다 (빨강 = 결함, 네이비 = 최종)
import gradio as gr

CARDS = [
    ('baseline', '베이스라인', '처음부터 학습한 Transformer · 6.0M · BLEU 4.87', '#8A8A8A'),
    ('naive', 'KoBART — EOS 결함', '학습 정답에 EOS 누락 · 124M · BLEU 5.21', '#990011'),
    ('final', 'KoBART — 최종', 'EOS 수정 + lr 2e-5 · 124M · BLEU 7.28', '#2F3C7E'),
]


def card_md(title, sub, color, res):
    head = (f"<div style='border-top:3px solid {color};padding:12px 4px'>"
            f"<b style='color:{color};font-size:15px'>{title}</b><br>"
            f"<span style='color:#9AA0A6;font-size:12px'>{sub}</span>")
    if res is None:
        return head + '</div>'
    end = '정상 종결' if res['ended'] else '⚠️ 문장이 안 끝남'
    end_c = '#3FA34D' if res['ended'] else '#E5484D'
    return (head +
            f"<div style='margin:14px 0;padding:14px;background:#EEF0F5;color:#1A1A1A;"
            f"border-radius:8px;font-size:16px;line-height:1.55;min-height:62px'>"
            f"{res['text'] or '(빈 응답)'}</div>"
            f"<span style='font-size:12px;color:#9AA0A6'>{res['len']}자 · </span>"
            f"<span style='font-size:12px;color:{end_c};font-weight:600'>{end}</span></div>")


def run(q, strategy):
    strategy = 'greedy' if strategy.startswith('greedy') else 'beam5'
    res = answer(q, strategy)
    if res is None:
        return ('', *[card_md(t, s, c, None) for _, t, s, c in CARDS])
    gt = ANSWER_MAP.get((q or '').strip())
    head = (f"**질문** {q.strip()}　　**데이터셋 정답** {gt}" if gt else
            f"**질문** {q.strip()}　　<span style='color:#9AA0A6'>"
            f"(test set에 없는 질문이라 정답 비교는 생략)</span>")
    return (head, *[card_md(t, s, c, res[k]) for k, t, s, c in CARDS])

print('UI 함수 준비 완료')

UI 함수 준비 완료


In [9]:
# 2026-09-21: Gradio Blocks 구성
EXAMPLES = ['오늘 생각보다 춥네', '청첩장을 찍는 날이 오다니', '배고파',
            '여자친구랑 헤어졌어', '시험 망한 것 같아', '월급 들어왔다']

with gr.Blocks(title='한국어 챗봇 응답 생성 — 모델 비교 데모') as demo:
    gr.Markdown(
        '# 한국어 챗봇 응답 생성 — 모델 비교 데모\n'
        '질문 하나를 세 모델에 동시에 넣어 답변을 비교합니다. '
        '`notebooks/final.ipynb`에서 학습한 체크포인트를 그대로 씁니다.\n\n'
        '> **여기서 보실 것** — 가운데 *EOS 결함* 모델은 문장을 끝내는 법을 배우지 못해 '
        '여러 문장이 이어지다 잘립니다. 오른쪽 최종 모델은 같은 질문에 한 문장으로 답합니다. '
        '학습 코드에서 **정답 끝에 EOS 토큰을 붙이는 한 줄**만 다릅니다.\n\n'
        '> 디코딩을 `greedy`로 바꾸면 왼쪽 베이스라인에서 **같은 말을 반복하는 붕괴**를 볼 수 있습니다. '
        '`beam5`가 이걸 잡아준다는 것이 6단계의 결론입니다.')

    with gr.Row():
        q_in = gr.Textbox(label='질문', placeholder='예) 오늘 생각보다 춥네', scale=5)
        strategy = gr.Radio(['beam5 (기본)', 'greedy'], value='beam5 (기본)',
                            label='디코딩 전략', scale=2)
        btn = gr.Button('답변 생성', variant='primary', scale=1)

    head_out = gr.Markdown()
    with gr.Row():
        outs = [gr.Markdown(card_md(t, s, c, None)) for _, t, s, c in CARDS]

    gr.Examples(examples=EXAMPLES, inputs=q_in, label='예시 질문')
    gr.Markdown(
        f"<span style='color:#9AA0A6;font-size:12px'>디코딩 — 베이스라인 beam5"
        f"(length_penalty 0.7) / KoBART beam5 + no_repeat_ngram 3 · "
        f"device: {device} · BLEU는 test 1,175건 기준</span>")

    # queue=False: SSE 대신 동기 POST 엔드포인트 사용 (위 마크다운 셀 설명 참고)
    for ev in (btn.click, q_in.submit):
        ev(run, [q_in, strategy], [head_out, *outs], queue=False)
    strategy.change(run, [q_in, strategy], [head_out, *outs], queue=False)

print('Blocks 구성 완료')

Blocks 구성 완료


## 6. 서버 실행

아래 셀을 실행하면 `http://127.0.0.1:7860` 에서 데모가 열린다. `prevent_thread_lock=True`라 셀이 바로 반환되므로 노트북을 계속 쓸 수 있고, **커널이 살아 있는 동안** 서버가 동작한다. 끄려면 맨 아래 `demo.close()` 셀을 실행한다.

In [10]:
# 2026-09-21: 서버 실행 (prevent_thread_lock=True - 셀이 바로 반환되어 노트북을 계속 쓸 수 있다)
demo.launch(server_name='127.0.0.1', server_port=7860,
            share=False, inbrowser=False, prevent_thread_lock=True)
print('브라우저에서 http://127.0.0.1:7860 을 열어보세요')

* Running on local URL:  http://127.0.0.1:7860


* To create a public link, set `share=True` in `launch()`.


브라우저에서 http://127.0.0.1:7860 을 열어보세요


In [11]:
# 2026-09-21: 서버 종료 (필요할 때 실행)
# demo.close()

## 요약

| 항목 | 내용 |
|---|---|
| 입력 | 질문 텍스트 + 디코딩 전략(beam5 / greedy) |
| 출력 | 세 모델의 답변 + 글자 수 + 정상 종결 여부, test set 질문이면 정답도 함께 |
| 모델 | 베이스라인(6.0M) + KoBART 결함(124M) + KoBART 최종(124M) |
| 추가 학습 | 없음 — `final.ipynb`가 저장한 체크포인트를 그대로 불러 쓴다 |

**이 데모로 보여줄 수 있는 것**

1. **전이학습의 효과** — 왼쪽(베이스라인)은 질문과 무관한 답을 자주 내고, 오른쪽(KoBART 최종)은 맥락에 맞는 한 문장을 낸다.
2. **EOS 결함** — 가운데는 여러 문장이 이어지다 중간에서 잘린다. 오른쪽과 **학습 코드 한 줄** 차이다.
3. **디코딩 전략의 효과** — `greedy`로 바꾸면 베이스라인에서 같은 말을 반복하는 붕괴가 재현된다.

**한계**: 세 모델을 동시에 올려 약 1GB의 GPU 메모리를 쓴다. CPU에서도 동작하지만 beam search가 느려진다.